# Hierarchical Clustering: Agglomerative & Divisive

This notebook covers:
1. Theory (bottom-up vs top-down)
2. **Agglomerative** clustering from scratch + with `scipy` / `scikit-learn`
3. Dendrograms and linkage methods (single, complete, average, ward)
4. Choosing the number of clusters (cophenetic correlation, silhouette)
5. **Divisive** clustering from scratch (DIANA) + `BisectingKMeans`

Requirements: `numpy matplotlib scipy scikit-learn` (scikit-learn >= 1.1).

## 1. Theory

| | Agglomerative (bottom-up) | Divisive (top-down) |
|---|---|---|
| Start | Each point is its own cluster | All points in one cluster |
| Step | Merge the two closest clusters | Split a cluster into two |
| Stop | One cluster remains (or k reached) | Singletons (or k reached) |
| Cost | ~O(n^2) to O(n^3) | Exhaustive split is O(2^n); heuristics (DIANA, bisecting k-means) used |

**Linkage criteria** (distance between clusters A and B):
- **Single**: min distance between any pair; can chain, finds elongated shapes
- **Complete**: max distance between any pair; compact clusters
- **Average**: mean of all pairwise distances
- **Ward**: merge that minimally increases total within-cluster variance (Euclidean only)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster, cophenet
from scipy.spatial.distance import pdist, squareform
from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import AgglomerativeClustering, BisectingKMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

np.random.seed(42)
plt.rcParams['figure.dpi'] = 100

## 2. Data

In [ ]:
X, y_true = make_blobs(n_samples=40, centers=3, cluster_std=1.0, random_state=7)

plt.figure(figsize=(5,4))
plt.scatter(X[:,0], X[:,1], s=40, edgecolor='k')
plt.title("Toy dataset (40 points)")
plt.show()

## 3. Agglomerative clustering from scratch

Algorithm:
1. Put every point in its own cluster.
2. Compute the distance between all cluster pairs using the chosen linkage.
3. Merge the closest pair; record the merge.
4. Repeat until `k` clusters remain.

In [ ]:
def cluster_distance(D, a, b, linkage_type):
    """Distance between clusters a and b (lists of point indices) using precomputed matrix D."""
    sub = D[np.ix_(a, b)]
    if linkage_type == 'single':   return sub.min()
    if linkage_type == 'complete': return sub.max()
    if linkage_type == 'average':  return sub.mean()
    raise ValueError(linkage_type)

def agglomerative(X, k=1, linkage_type='average'):
    D = squareform(pdist(X))
    clusters = {i: [i] for i in range(len(X))}     # cluster id -> members
    history = []                                   # (id1, id2, distance, new_size)
    next_id = len(X)
    while len(clusters) > k:
        ids = list(clusters)
        best = (None, None, np.inf)
        for i in range(len(ids)):
            for j in range(i+1, len(ids)):
                d = cluster_distance(D, clusters[ids[i]], clusters[ids[j]], linkage_type)
                if d < best[2]:
                    best = (ids[i], ids[j], d)
        a, b, d = best
        clusters[next_id] = clusters.pop(a) + clusters.pop(b)
        history.append((a, b, d, len(clusters[next_id])))
        next_id += 1
    labels = np.empty(len(X), dtype=int)
    for lab, members in enumerate(clusters.values()):
        labels[members] = lab
    return labels, history

for link in ['single', 'complete', 'average']:
    labels, hist = agglomerative(X, k=3, linkage_type=link)
    sk = AgglomerativeClustering(n_clusters=3, linkage=link).fit_predict(X)
    print(f"{link:9s} ARI vs sklearn = {adjusted_rand_score(labels, sk):.3f}   (1.0 = identical)")

In [ ]:
labels, hist = agglomerative(X, k=3, linkage_type='average')
plt.figure(figsize=(5,4))
plt.scatter(X[:,0], X[:,1], c=labels, cmap='viridis', s=50, edgecolor='k')
plt.title("From-scratch agglomerative (average linkage, k=3)")
plt.show()

print("Last 5 merges (id1, id2, distance, new cluster size):")
full_labels, full_hist = agglomerative(X, k=1, linkage_type='average')
for h in full_hist[-5:]:
    print(h[0], h[1], round(h[2], 3), h[3])

## 4. Dendrograms with SciPy

A dendrogram shows the merge order; the **height** is the distance at which two clusters merged. Cutting horizontally gives a flat clustering.

In [ ]:
methods = ['single', 'complete', 'average', 'ward']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, m in zip(axes.ravel(), methods):
    Z = linkage(X, method=m)
    dendrogram(Z, ax=ax, no_labels=True, color_threshold=0.7*max(Z[:,2]))
    ax.set_title(f"{m} linkage")
    ax.set_ylabel("distance")
plt.tight_layout()
plt.show()

In [ ]:
Z = linkage(X, method='ward')
cut_height = 8
flat = fcluster(Z, t=3, criterion='maxclust')    # exactly 3 clusters

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
dendrogram(Z, ax=ax[0], no_labels=True)
ax[0].axhline(Z[-3, 2] + 0.01, color='r', ls='--', label='cut for k=3')
ax[0].legend(); ax[0].set_title("Ward dendrogram")
ax[1].scatter(X[:,0], X[:,1], c=flat, cmap='viridis', s=50, edgecolor='k')
ax[1].set_title("Flat clusters (k=3)")
plt.show()

## 5. Choosing the number of clusters

- **Largest gap** between successive merge heights.
- **Silhouette score** for each candidate k.
- **Cophenetic correlation** measures how faithfully the dendrogram preserves original pairwise distances (closer to 1 is better).

In [ ]:
Z = linkage(X, method='ward')
heights = Z[:, 2]
gaps = np.diff(heights)
print("Largest gap suggests k =", len(X) - (np.argmax(gaps) + 1))

ks = range(2, 9)
sil = [silhouette_score(X, fcluster(Z, t=k, criterion='maxclust')) for k in ks]
plt.figure(figsize=(5,3.5))
plt.plot(list(ks), sil, 'o-')
plt.xlabel("k"); plt.ylabel("silhouette"); plt.title("Silhouette vs k (Ward)")
plt.show()
print("Best k by silhouette:", list(ks)[int(np.argmax(sil))])

print("\nCophenetic correlation:")
for m in methods:
    c, _ = cophenet(linkage(X, m), pdist(X))
    print(f"  {m:9s} {c:.3f}")

## 6. Linkage behaviour on non-spherical data

Single linkage can follow chains (good for moons); Ward/complete favour round clusters.

In [ ]:
Xm, _ = make_moons(n_samples=200, noise=0.06, random_state=0)
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
for ax, m in zip(axes, methods):
    lab = AgglomerativeClustering(n_clusters=2, linkage=m).fit_predict(Xm)
    ax.scatter(Xm[:,0], Xm[:,1], c=lab, cmap='coolwarm', s=18)
    ax.set_title(m)
plt.tight_layout(); plt.show()

## 7. Divisive clustering from scratch (DIANA)

**DIANA** (DIvisive ANAlysis):
1. Start with all points in one cluster.
2. Pick the cluster with the largest diameter (max pairwise distance).
3. Find the point with the highest average dissimilarity to the others; it starts the *splinter group*.
4. Repeatedly move points whose average distance to the splinter group is **smaller** than to the remaining points.
5. Stop moving when no point qualifies; this split gives two clusters. Repeat until `k` clusters.

In [ ]:
def diana(X, k=3):
    D = squareform(pdist(X))
    clusters = [list(range(len(X)))]
    splits = []
    while len(clusters) < k:
        # choose splittable cluster with largest diameter
        diam = [D[np.ix_(c, c)].max() if len(c) > 1 else -1 for c in clusters]
        ci = int(np.argmax(diam))
        c = clusters.pop(ci)
        sub = D[np.ix_(c, c)]
        # seed the splinter group
        avg = sub.sum(axis=1) / (len(c) - 1)
        seed = int(np.argmax(avg))
        splinter, rest = [seed], [i for i in range(len(c)) if i != seed]
        moved = True
        while moved and len(rest) > 1:
            moved = False
            best_gain, best_i = 0, None
            for i in rest:
                d_rest = sub[i, [j for j in rest if j != i]].mean()
                d_spl  = sub[i, splinter].mean()
                if d_rest - d_spl > best_gain:
                    best_gain, best_i = d_rest - d_spl, i
            if best_i is not None:
                rest.remove(best_i); splinter.append(best_i); moved = True
        A = [c[i] for i in splinter]; B = [c[i] for i in rest]
        splits.append((len(c), len(A), len(B), diam[ci]))
        clusters += [A, B]
    labels = np.empty(len(X), dtype=int)
    for lab, members in enumerate(clusters):
        labels[members] = lab
    return labels, splits

lab_diana, splits = diana(X, k=3)
print("Splits (parent size, child A, child B, parent diameter):")
for s in splits: print("  ", s[0], "->", s[1], "+", s[2], f"(diameter {s[3]:.2f})")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(X[:,0], X[:,1], c=lab_diana, cmap='viridis', s=50, edgecolor='k'); ax[0].set_title("DIANA (k=3)")
ward = AgglomerativeClustering(n_clusters=3, linkage='ward').fit_predict(X)
ax[1].scatter(X[:,0], X[:,1], c=ward, cmap='viridis', s=50, edgecolor='k'); ax[1].set_title("Agglomerative Ward (k=3)")
plt.show()
print("ARI DIANA vs Ward:", round(adjusted_rand_score(lab_diana, ward), 3))
print("ARI DIANA vs truth:", round(adjusted_rand_score(lab_diana, y_true), 3))

## 8. Divisive with Bisecting K-Means (scikit-learn)

A scalable top-down method: repeatedly split a cluster with 2-means. Scikit-learn exposes the idea via `BisectingKMeans`; setting `bisecting_strategy='largest_cluster'` splits the biggest cluster, `'biggest_inertia'` (default) the loosest one.

In [ ]:
bkm = BisectingKMeans(n_clusters=3, random_state=0, bisecting_strategy='biggest_inertia')
lab_bkm = bkm.fit_predict(X)

plt.figure(figsize=(5,4))
plt.scatter(X[:,0], X[:,1], c=lab_bkm, cmap='viridis', s=50, edgecolor='k')
plt.scatter(*bkm.cluster_centers_.T, c='red', marker='X', s=200, edgecolor='k')
plt.title("Bisecting K-Means (k=3)")
plt.show()
print("ARI vs truth:", round(adjusted_rand_score(lab_bkm, y_true), 3))

## 9. Side-by-side comparison

In [ ]:
results = {
    "Agglo single":   AgglomerativeClustering(3, linkage='single').fit_predict(X),
    "Agglo complete": AgglomerativeClustering(3, linkage='complete').fit_predict(X),
    "Agglo average":  AgglomerativeClustering(3, linkage='average').fit_predict(X),
    "Agglo ward":     ward,
    "Divisive DIANA": lab_diana,
    "Bisecting KMeans": lab_bkm,
}
print(f"{'Method':18s} {'Silhouette':>10s} {'ARI vs truth':>13s}")
for name, lab in results.items():
    print(f"{name:18s} {silhouette_score(X, lab):10.3f} {adjusted_rand_score(y_true, lab):13.3f}")

## 10. Summary

- **Agglomerative** is deterministic, gives a full dendrogram, and is the standard choice; cost grows quickly with n (memory O(n^2)).
- **Divisive** can be more accurate at the top of the hierarchy (it sees global structure first) but is costly; DIANA and bisecting k-means are practical heuristics.
- Linkage matters: **single** = chaining/non-convex shapes, **complete/average/ward** = compact clusters. Always scale features first for real data.
- Pick k using dendrogram gaps, silhouette, and domain knowledge.

**Exercises:** try `make_blobs` with overlapping clusters, add outliers and see how single linkage reacts, or cluster a real dataset (e.g. `load_iris`) after `StandardScaler`.